# Long-lived eddies: depth-averaged snapshot case studies

This is the depth-averaged companion to `long_eddies_depth.ipynb`. It loads the cached `source='depth_snapshot'` table, which contains one thickness-weighted, column-averaged row per eddy-day. The panels therefore show one curve per variable rather than separate fixed-depth curves. Blue and orange background shading identify planetary- and topographic-beta-dominated snapshots.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
import seacofs_tilt_tools as tilt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)

## Settings and cached data

In [ ]:
DOMINANCE_FACTOR = 2.0
N_PER_POLARITY = 15

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
snapshot_df = tilt.add_pv_gradient_terms(source='depth_snapshot')

assert not snapshot_df.duplicated(['Eddy', 'Day']).any()
print(f'{len(snapshot_df):,} depth-averaged snapshots')

## Select the longest-lived eddies

Selection uses one depth-averaged row per eddy-day. If `Age` is unavailable, observed day span is used.

In [ ]:
if 'Age' in snapshot_df:
    lifetime = snapshot_df.groupby(['Cyc', 'Eddy']).Age.first()
else:
    lifetime = snapshot_df.groupby(['Cyc', 'Eddy']).Day.agg(np.ptp)
selected_ids = (lifetime.groupby(level='Cyc').nlargest(N_PER_POLARITY)
                .index.get_level_values('Eddy').unique())
selected = snapshot_df[snapshot_df.Eddy.isin(selected_ids)].copy()
display(lifetime.loc[lifetime.index.get_level_values('Eddy').isin(selected_ids)]
        .rename('lifetime_days').reset_index()
        .sort_values(['Cyc', 'lifetime_days'], ascending=[True, False]))

## Depth-averaged snapshot time plot

The row arrangement matches the depth-resolved notebook, but every PV-related value is the cached depth-following column average for that snapshot. Regime shading is calculated from the same rows being plotted.

In [ ]:
def time_plot_depth_snapshot(eddy, snapshot_data=snapshot_df, grid=grid, clip_outliers=True):
    snap = snapshot_data[snapshot_data.Eddy.eq(eddy)].sort_values('Day').copy()
    if snap.empty:
        raise ValueError(f'Eddy {eddy} is not present in snapshot_data')
    snap['t'] = snap.Day - snap.Day.iloc[0]
    cyc = snap.Cyc.iloc[0]

    EDDY_COLOR = 'tab:red' if cyc == 'AE' else 'tab:blue'
    PLAN_COLOR, TOPO_COLOR = 'tab:blue', 'tab:orange'
    threshold = np.log(DOMINANCE_FACTOR)
    planetary = snap.topo_plan_ratio_smooth <= -threshold
    topographic = snap.topo_plan_ratio_smooth >= threshold

    fig = plt.figure(figsize=(13, 11))
    gs = fig.add_gridspec(6, 2, width_ratios=[2.2, 1])
    axs = [fig.add_subplot(gs[i, 0]) for i in range(6)]
    axm = fig.add_subplot(gs[:, 1])
    for ax in axs:
        for t in snap.loc[planetary, 't']:
            ax.axvspan(t-.5, t+.5, color=PLAN_COLOR, alpha=.2, lw=0)
        for t in snap.loc[topographic, 't']:
            ax.axvspan(t-.5, t+.5, color=TOPO_COLOR, alpha=.2, lw=0)

    axs[0].plot(snap.t, snap.TiltDis, color='magenta', lw=1.7)
    axs[0].set_ylabel('Tilt distance [km]', color='magenta')
    axs[0].tick_params(axis='y', labelcolor='magenta')
    ax0 = axs[0].twinx()
    ax0.plot(snap.t, snap.PV_grad_mag, color=EDDY_COLOR, lw=1.3)
    ax0.set_ylabel(r'$|\nabla PV|$', color=EDDY_COLOR)
    ax0.tick_params(axis='y', labelcolor=EDDY_COLOR)

    axs[1].plot(snap.t, snap.topo_plan_ratio, color=EDDY_COLOR, lw=1.3)
    axs[1].axhline(0, color='0.2', lw=.8)
    axs[1].axhline(threshold, color=TOPO_COLOR, ls='--', lw=.8)
    axs[1].axhline(-threshold, color=PLAN_COLOR, ls='--', lw=.8)
    axs[1].set_ylabel(r'$\ln(|\nabla PV|_{topo}/|\nabla PV|_{plan})$')

    axs[2].plot(snap.t, snap.Ro, color=EDDY_COLOR, lw=1.3)
    axs[2].axhline(1, color='0.3', ls=':', lw=.8)
    axs[2].set_ylabel(r'$Ro$')

    axs[3].plot(snap.t, snap.w, color=EDDY_COLOR, lw=1.3)
    axs[3].set_ylabel(r'$\zeta$')

    axs[4].plot(snap.t, snap.h/1e3, color=EDDY_COLOR, lw=1.3)
    axs[4].set_ylabel('Depth [km]')

    axs[5].plot(snap.t, snap.PV, color=EDDY_COLOR, lw=1.3)
    axs[5].set(xlabel='Eddy age [days]', ylabel='PV')

    if clip_outliers:
        limits = {
            ax0: snap.PV_grad_mag,
            axs[1]: snap.topo_plan_ratio,
            axs[2]: snap.Ro,
            axs[3]: snap.w,
            axs[4]: snap.h/1e3,
            axs[5]: snap.PV,
        }
        for ax, values in limits.items():
            values = values[np.isfinite(values)]
            if len(values):
                lo, hi = np.nanpercentile(values, [5, 95])
                full_range = values.max() - values.min()
                core_range = hi - lo
                if core_range > 0 and full_range > 3*core_range:
                    ax.set_ylim(lo, hi)

    axs[4].invert_yaxis()

    for ax in axs:
        ax.grid(alpha=.2)
        ax.margins(x=0)

    map_rows = snap.dropna(subset=['xc', 'yc']).copy()
    if map_rows.empty:
        raise ValueError(f'Eddy {eddy} has no valid centroid coordinates')
    pad = 20
    xmin, xmax = map_rows.xc.min()-pad, map_rows.xc.max()+pad
    ymin, ymax = map_rows.yc.min()-pad, map_rows.yc.max()+pad
    inside = ((grid.X_grid >= xmin) & (grid.X_grid <= xmax) &
              (grid.Y_grid >= ymin) & (grid.Y_grid <= ymax))
    bathy = np.where((grid.mask_rho == 1) & inside, grid.h/1e3, np.nan)
    cf = axm.contourf(grid.X_grid, grid.Y_grid, bathy, cmap='Greys_r')
    fig.colorbar(cf, ax=axm, orientation='horizontal', location='bottom',
                 label='Depth [km]', shrink=.8, pad=.08)
    axm.plot(map_rows.xc, map_rows.yc, '.-', color=EDDY_COLOR,
             lw=1.5, ms=3, label='Surface centre')
    map_rows['day_idx'] = map_rows.Day - map_rows.Day.min()
    for day in np.arange(50, map_rows.day_idx.max()+1, 50):
        part = map_rows[map_rows.day_idx.eq(day)]
        if not part.empty:
            point = part.iloc[0]
            axm.annotate(f'D{day}', (point.xc, point.yc), xytext=(4, 4),
                         textcoords='offset points', fontsize=8)
    axm.set(xlim=(xmin, xmax), ylim=(ymin, ymax),
            xlabel='x [km]', ylabel='y [km]', title=f'{cyc}{eddy}')
    axm.set_aspect('equal')
    axm.legend(fontsize=7, frameon=False)
    plt.tight_layout()
    plt.show()
    return fig, axs, axm

## Generate the selected cases

In [ ]:
for eddy in selected.Eddy.unique():
    time_plot_depth_snapshot(eddy, selected, grid)